# Mutation sensitivity and interface interpretability

**Status:** evaluation scaffold for review.

**When to run:** after a residue-level model has been trained and frozen.

**Inputs:** a model-specific scorer, wild-type PPI pairs, residue positions,
and independently sourced interface annotations. Optional experimental
mutation labels enable direction/impact validation.

**Outputs:** mutation score deltas, interface-enrichment statistics, and
residue plots under the drafting output folders when enabled.

**Usage rule:** attention weights are hypotheses, not explanations by
themselves. Evaluate enrichment against held-out interface annotations and
compare score changes with experimental mutations where available.

**Shared protocol:** set `SNOOPPI_ROOT` and `SNOOPPI_SPLIT_DIR` before setup.
Use the same split directory in every draft. Existing caches use original raw
sequence keys; this suite does not regenerate embeddings. Metadata uses full
sequence hashes, while leakage audits also check normalized 1,024-residue inputs.


## 1. Setup and file contracts

In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import mannwhitneyu, spearmanr

WRITE_OUTPUTS = False
RUN_TAG = "mutation_interface_v1"
MUTATION_TABLE_PATH = PATHS["root"] / "raw" / "mutation_benchmark.csv"
INTERFACE_TABLE_PATH = PATHS["root"] / "raw" / "interface_annotations.csv"

MUTATION_COLUMNS = {
    "pair_id", "partner", "position_1based", "wild_type", "mutant",
    "sequence_A", "sequence_B", "evidence_source",
}
INTERFACE_COLUMNS = {"pair_id", "partner", "position_1based", "is_interface", "evidence_source"}
CHECKPOINT_PATH = None  # Set to the frozen scorer checkpoint before RUN_SCORING=True.

print("Mutation input:", MUTATION_TABLE_PATH)
print("Interface input:", INTERFACE_TABLE_PATH)

## 2. Validate annotations and construct mutants
Annotation `pair_id` is the unordered **full-sequence** pair hash from
`full_pair_id`, with raw partner orientation retained for A/B positions.
Wild-type pairs must belong to the selected test partition. Positions are
1-based on uppercase, space-removed sequences, within the first 1,024 residues.
Annotations require independent `evidence_source` values. This notebook checks
input contracts; checkpoint data provenance must also be reviewed.


In [ ]:
if not MUTATION_TABLE_PATH.exists():
    raise FileNotFoundError(MUTATION_TABLE_PATH)
mutations = pd.read_csv(MUTATION_TABLE_PATH)
missing = MUTATION_COLUMNS - set(mutations)
if missing:
    raise ValueError(f"Mutation table missing: {sorted(missing)}")

def apply_substitution(sequence, position_1based, wild_type, mutant):
    sequence = str(sequence).upper().replace(" ", "")
    numeric_position = float(position_1based)
    if not np.isfinite(numeric_position) or not numeric_position.is_integer():
        raise ValueError("Mutation position must be a finite integer")
    index = int(numeric_position) - 1
    wild_type, mutant = str(wild_type).upper(), str(mutant).upper()
    alphabet = set("ACDEFGHIKLMNPQRSTVWY")
    if wild_type not in alphabet or mutant not in alphabet or wild_type == mutant:
        raise ValueError("Expected a single canonical amino-acid substitution")
    if index >= 1024:
        raise ValueError("Mutation lies outside the 1,024-residue cached input window")
    if index < 0 or index >= len(sequence):
        raise IndexError(f"Position {position_1based} outside sequence length {len(sequence)}")
    if sequence[index] != wild_type:
        raise ValueError(
            f"Annotation expects {wild_type} at {position_1based}, found {sequence[index]}"
        )
    return sequence[:index] + mutant + sequence[index + 1:]

splits = load_splits(PATHS)
assert_model_ready(splits)
manifest = split_manifest(PATHS)
heldout_pair_ids = set(prediction_identity(splits["test"], manifest).full_pair_id)
if mutations.evidence_source.isna().any() or mutations.evidence_source.astype(str).str.strip().eq("").any():
    raise ValueError("Mutation annotations require independent evidence provenance")

mutant_rows = []
for row in mutations.itertuples(index=False):
    sequence_a, sequence_b = row.sequence_A, row.sequence_B
    if not all(isinstance(s, str) and s.strip() for s in (sequence_a, sequence_b)):
        raise ValueError("Mutation wild-type sequences must be nonempty strings")
    expected_pair = full_pair_id(sequence_a, sequence_b)
    if row.pair_id != expected_pair or expected_pair not in heldout_pair_ids:
        raise ValueError("Mutation pair_id must match full_pair_id and belong to the selected held-out test partition")
    if row.partner == "A":
        mutant_a = apply_substitution(sequence_a, row.position_1based, row.wild_type, row.mutant)
        mutant_b = sequence_b
    elif row.partner == "B":
        mutant_a = sequence_a
        mutant_b = apply_substitution(sequence_b, row.position_1based, row.wild_type, row.mutant)
    else:
        raise ValueError("partner must be A or B")
    mutant_rows.append({**row._asdict(), "mutant_sequence_A": mutant_a, "mutant_sequence_B": mutant_b})
mutant_pairs = pd.DataFrame(mutant_rows)
display(mutant_pairs.head())

## 3. Connect a frozen scorer

Implement `score_pairs` as a thin adapter around the selected checkpoint.
It must return one interaction probability per `(sequence_A, sequence_B)`
pair and must not update model weights. Keeping the adapter explicit avoids
mixing model-specific tokenization into the evaluation logic.

In [ ]:
def score_pairs(sequence_pairs):
    """Replace this body with a frozen-model inference adapter.

    Expected input: list of (sequence_A, sequence_B) strings.
    Expected output: one-dimensional NumPy array in [0, 1].
    """
    raise NotImplementedError("Connect the frozen model checkpoint before running mutation scoring")


RUN_SCORING = False
if RUN_SCORING:
    if CHECKPOINT_PATH is None or not Path(CHECKPOINT_PATH).is_file():
        raise FileNotFoundError("Set CHECKPOINT_PATH to the frozen scorer checkpoint")
    checkpoint_sha256 = file_sha256(Path(CHECKPOINT_PATH))
    wild_type_pairs = list(zip(mutant_pairs["sequence_A"], mutant_pairs["sequence_B"]))
    altered_pairs = list(zip(mutant_pairs["mutant_sequence_A"], mutant_pairs["mutant_sequence_B"]))
    mutant_pairs["wild_type_score"] = score_pairs(wild_type_pairs)
    mutant_pairs["mutant_score"] = score_pairs(altered_pairs)
    mutant_pairs["delta_score"] = mutant_pairs["mutant_score"] - mutant_pairs["wild_type_score"]
    if not mutant_pairs[["wild_type_score", "mutant_score"]].apply(lambda column: column.between(0, 1).all()).all():
        raise ValueError("Scorer returned values outside [0, 1]")
    display(mutant_pairs.sort_values("delta_score").head(20))
else:
    print("Scoring disabled. Implement score_pairs, verify the checkpoint, then set RUN_SCORING=True.")

## 4. Compare sensitivity with interface annotations

In [ ]:
if RUN_SCORING:
    if not INTERFACE_TABLE_PATH.exists():
        raise FileNotFoundError(INTERFACE_TABLE_PATH)
    interfaces = pd.read_csv(INTERFACE_TABLE_PATH)
    missing = INTERFACE_COLUMNS - set(interfaces)
    if missing:
        raise ValueError(f"Interface table missing: {sorted(missing)}")
    if not interfaces.is_interface.isin([0, 1]).all():
        raise ValueError("is_interface must be binary")
    if interfaces.evidence_source.isna().any() or interfaces.evidence_source.astype(str).str.strip().eq("").any():
        raise ValueError("Interface annotations require evidence provenance")
    evaluation = mutant_pairs.merge(
        interfaces,
        on=["pair_id", "partner", "position_1based"],
        how="inner",
        validate="many_to_one",
        suffixes=("_mutation", "_interface"),
    )
    evaluation["absolute_delta"] = evaluation["delta_score"].abs()
    interface_delta = evaluation.loc[evaluation["is_interface"].eq(1), "absolute_delta"]
    noninterface_delta = evaluation.loc[evaluation["is_interface"].eq(0), "absolute_delta"]
    if interface_delta.empty or noninterface_delta.empty:
        raise ValueError("Both interface and non-interface residues are required")
    print("Exploratory residue-level test: repeated mutations/pairs violate independence; use pair/group resampling for inference.")
    statistic, p_value = mannwhitneyu(interface_delta, noninterface_delta, alternative="greater")
    enrichment_summary = pd.DataFrame([{
        "interface_mutations": len(interface_delta),
        "noninterface_mutations": len(noninterface_delta),
        "interface_median_absolute_delta": interface_delta.median(),
        "noninterface_median_absolute_delta": noninterface_delta.median(),
        "mannwhitney_u": statistic,
        "one_sided_p_value": p_value,
    }])
    display(enrichment_summary.round(4))

    if "experimental_effect" in evaluation:
        correlation, correlation_p = spearmanr(evaluation["delta_score"], evaluation["experimental_effect"], nan_policy="omit")
        print("Spearman model-vs-experimental effect:", correlation, "p=", correlation_p)

## 5. Residue-level visualization and optional outputs

In [ ]:
if RUN_SCORING:
    selected_pair = evaluation["pair_id"].iloc[0]
    selected_partner = evaluation["partner"].iloc[0]
    selected = evaluation.loc[evaluation["pair_id"].eq(selected_pair) & evaluation["partner"].eq(selected_partner)].sort_values("position_1based")
    figure, axis = plt.subplots(figsize=(10, 4.5))
    for flag, label, color, marker in [(1, "Interface", "#D58632", "o"),
                                       (0, "Non-interface", "#176B87", "s")]:
        points = selected.loc[selected["is_interface"].eq(flag)]
        axis.scatter(points["position_1based"], points["delta_score"], color=color,
                     marker=marker, alpha=0.8, label=label)
    axis.legend(frameon=False)
    axis.axhline(0, color="#333333", linewidth=1)
    axis.set(
        title=f"In-silico mutation effects: {selected_pair[:12]} · partner {selected_partner}",
        xlabel="Residue position (1-based)",
        ylabel="Mutant − wild-type interaction score",
    )
    figure.tight_layout()
    plt.show()

    if WRITE_OUTPUTS:
        table_path = PATHS["tables"] / f"{RUN_TAG}_mutation_scores.csv"
        figure_path = PATHS["figures"] / f"{RUN_TAG}_{selected_pair}_mutation_scan.png"
        manifest_path = PATHS["tables"] / f"{RUN_TAG}_manifest.json"
        prepare_outputs([table_path, figure_path, manifest_path])
        save_json(make_run_manifest(PATHS, splits, {"run_tag": RUN_TAG, "checkpoint_sha256": checkpoint_sha256,
                  "mutation_input_sha256": file_sha256(MUTATION_TABLE_PATH),
                  "interface_input_sha256": file_sha256(INTERFACE_TABLE_PATH)}), manifest_path)
        evaluation.to_csv(table_path, index=False)
        figure.savefig(figure_path, dpi=300, bbox_inches="tight")
else:
    print("No figures or outputs until RUN_SCORING=True.")

## Interpretation limits

- Use annotations not seen in training or model selection.
- Correct for repeated mutations from the same protein/pair in uncertainty
  estimates; pair-level bootstrap is preferable to treating residues as
  independent.
- A predicted score delta is not a binding free-energy estimate.
- Prioritize a small, preregistered mutation set for literature or wet-lab
  validation rather than selecting only visually convincing examples.